https://docs.opendota.com/

In [1]:
import json
from dataclasses import dataclass
import datetime
import pathlib

import plotly.graph_objects as go
import requests
import numpy as np

In [2]:
HEROES_FILE = pathlib.Path('heroes.json')

with open(HEROES_FILE) as f:
    HEROES_JSON = json.load(f)

HEROES = {int(heroidx): hero for heroidx, hero in HEROES_JSON.items()}

In [3]:
PLAYER_REQ = 'https://api.opendota.com/api/players/{account_id}'

def get_player(account_id):
    response = requests.get(PLAYER_REQ.format(account_id=account_id))
    if response.status_code != 200:
        return {}
    return response.json()


MATCHES_REQ = 'https://api.opendota.com/api/players/{account_id}/matches'

def get_matches(account_id):
    response = requests.get(MATCHES_REQ.format(account_id=account_id))
    if response.status_code != 200:
        return []
    return response.json()

In [4]:
AIRAT = 195123691
LION = 157739135
STAS = 1087711740
AZAT = 1019698762

In [5]:
airat_stats = get_player(AIRAT)
lion_stats = get_player(LION)
stas_stats = get_player(STAS)
azat_stats = get_player(AZAT)

In [12]:
@dataclass
class Match:
    match_id: int
    start_time: datetime.datetime
    duration: datetime.timedelta
    win: bool
    is_radiant: bool
    hero_name: str
    kda: tuple[int, int, int]
    avg_rank: int

    @classmethod
    def from_json(cls, data):
        # player_slot: 0-127 are Radiant, 128-255 are Dire
        is_radiant = data['player_slot'] < 128
        win = data['radiant_win'] == is_radiant
        return cls(
            match_id=data['match_id'],
            start_time=datetime.datetime.fromtimestamp(data['start_time']),
            duration=datetime.timedelta(seconds=data['duration']),
            win=win,
            is_radiant=is_radiant,
            hero_name=HEROES[data['hero_id']]['localized_name'],
            kda=(data['kills'], data['deaths'], data['assists']),
            avg_rank=data['average_rank'],
        )
    
    def calc_kda(self):
        k, d, a = self.kda
        if d == 0:
            d = .75
        return (k + a) / d
    
    def __str__(self):
        return f'({"-+"[self.win]})[{self.start_time:%H:%M %d.%m.%y}, {self.duration.seconds/60:.0f}min] {"/".join(map(str, self.kda))} "{self.hero_name}"'


In [64]:
class Analysis:
    def __init__(self, account_id):
        self.account_id = account_id
        self.player = get_player(account_id)
        if not self.player:
            print(f'No player found {account_id}')
        matches_resp = get_matches(account_id)
        self.matches = [Match.from_json(match) for match in reversed(matches_resp) if match['hero_id']]
    
    def get_match_duration_hist(self):
        durations = [match.duration.seconds for match in self.matches]
        fig = go.Figure(data=[go.Histogram(x=[d / 60 for d in durations], nbinsx=35)])
        fig.update_layout(title_text='Match duration histogram')
        fig.update_layout(template='plotly_dark')
        return fig
    
    def get_hero_statistic(self):
        hero_stats = {}
        for match in self.matches:
            hero_name = match.hero_name
            if hero_name not in hero_stats:
                hero_stats[hero_name] = {'played': 0, 'won': 0}
            hero_stats[hero_name]['played'] += 1
            if match.win:
                hero_stats[hero_name]['won'] += 1
        hero_stats = {hero_name: {'played': stats['played'], 'winrate': stats['won'] / stats['played']} for hero_name, stats in hero_stats.items()}
        hero_stats = {hero_name: stats for hero_name, stats in sorted(hero_stats.items(), key=lambda x: x[1]['played'], reverse=True)}
        hero_stats_short = {hero_name: stats for hero_name, stats in hero_stats.items() if stats['played'] > 9}
        fig = go.Figure(data=[go.Bar(x=list(hero_stats_short.keys()), y=[stats['played'] for stats in hero_stats_short.values()], marker_color=[stats['winrate'] for stats in hero_stats_short.values()])])
        fig.update_traces(hovertemplate='played: %{y}<br>winrate: %{marker.color:.1%}')
        fig.update_coloraxes(colorscale='Viridis', colorbar_title='Winrate')
        fig.update_layout(title_text='Hero statistics')
        fig.update_layout(template='plotly_dark')
        return fig

    def get_winrate_sliding_window(self):
        times = [match.start_time for match in self.matches]
        window = 50
        winlose = [int(match.win) for match in self.matches]
        colors = ['green' if wl else 'red' for wl in winlose]
        winrate = [np.mean(winlose[i-window:i]) for i in range(window, len(winlose))]
        fig = go.Figure(data=[go.Scatter(x=times[window:], y=winrate, mode='lines+markers', marker=dict(color=colors[window:]))])
        fig.update_layout(title_text='Winrate vs time')
        fig.update_layout(template='plotly_dark')
        return fig

In [14]:
lion = Analysis(LION)

In [65]:
airat = Analysis(AIRAT)

In [44]:
airat.matches[-1]

Match(match_id=7717743442, start_time=datetime.datetime(2024, 5, 3, 0, 22, 37), duration=datetime.timedelta(seconds=2534), win=False, is_radiant=False, hero_name='Disruptor', kda=(4, 9, 17), avg_rank=52)

In [15]:
airat.get_match_duration_hist().show()

In [45]:
airat.get_hero_statistic().show()

In [66]:
airat.get_winrate_sliding_window().show()

In [9]:
azat = Analysis(AZAT)

No player found 1019698762


In [28]:
fig = lion.get_winrate_sliding_window()
fig.show()

## Learning a model to predict the outcome of a Dota 2 match given kda

In [67]:
import pandas as pd

data = []
for match in lion.matches:
    data.append([match.kda[0], match.kda[1], match.kda[2], match.win])

df = pd.DataFrame(data, columns=['kills', 'deaths', 'assists', 'win'])

df

,kills,deaths,assists,win
0,7,13,14,False
1,13,12,20,True
2,0,6,11,False
3,5,8,10,False
4,5,5,17,True
...,...,...,...,...
6349,4,4,9,True
6350,24,4,13,True
6351,5,16,14,False
6352,2,13,6,False


In [68]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# standardize kda
df['kills'] = (df['kills'] - df['kills'].mean()) / df['kills'].std()
df['deaths'] = (df['deaths'] - df['deaths'].mean()) / df['deaths'].std()
df['assists'] = (df['assists'] - df['assists'].mean()) / df['assists'].std()

# use kda and hero to predict win
# one-hot encode hero
# df = pd.get_dummies(df, columns=['hero'])

X = df.drop(columns=['win'])
y = df['win']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# model = RandomForestClassifier()
model = SVC()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

accuracy_score(y_test, y_pred)

0.7726199842643587

In [76]:
# 3d kda scatter plot
fig = go.Figure(data=[go.Scatter3d(x=df['kills'], y=df['deaths'], z=df['assists'], mode='markers', 
    marker=dict(color=['green' if w else 'red' for w in df['win']], size=1))])
fig.update_layout(title_text='KDA scatter plot')
fig.update_layout(template='plotly_dark')
fig.show()

In [74]:
model.predict([[0, 0, 0]])

c:\Users\Airat\torchvenv\Lib\site-packages\sklearn\base.py:465: UserWarning:

X does not have valid feature names, but SVC was fitted with feature names



array([False])